# LiciGuard · Construir la base de conocimiento (RAG) automáticamente

Este cuaderno hace **todo solo**:

```
1. Descarga los PDF oficiales  →  2. Saca el texto  →  3. Lo corta por artículos  →  4. Crea articulos.json
   (ley, reglamento, normas)         (página por página)    (cada artículo = 1 ficha)       (lo que lee la app)
```

Al final obtienes **un solo archivo**: `articulos.json`. Lo pones en la carpeta `data/normativa/` de tu proyecto y la app ya sabe "buscar en la ley" cuando detecta una señal o cuando el usuario pregunta algo.

**Cómo usarlo:** ábrelo en Google Colab y ejecuta las celdas en orden con `Shift + Enter`. No tienes que descargar nada a mano.

In [ ]:
# Paso 1 · Instalar lo necesario (tarda unos segundos)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf", "requests"], check=True)
print("Listo")

## Paso 2 · Lista de documentos a descargar

Ya viene con los documentos principales. Puedes **agregar más** copiando una línea y cambiando el nombre y el enlace (por ejemplo, opiniones del OECE o la directiva de bases estándar).

> Usa de preferencia enlaces oficiales (Congreso, El Peruano, gob.pe). Si un enlace deja de funcionar, búscalo de nuevo y reemplázalo.

In [ ]:
DOCUMENTOS = [
    # (nombre que verá el usuario,                        enlace al PDF)
    ("Ley N.° 32069 (texto consolidado, Congreso)",       "https://leyes.congreso.gob.pe/Documentos/2021_2026/ADLP/Texto_Consolidado/32069-TXM.pdf"),
    ("Reglamento de la Ley N.° 32069 (D.S. 009-2025-EF)", "https://img.lpderecho.pe/wp-content/uploads/2025/01/Reglamento-de-la-Ley-32069-Ley-General-de-Contrataciones-Publicas-LP-DERECHO.pdf"),
    ("Modificación del Reglamento (D.S. 001-2026-EF)",    "https://img.lpderecho.pe/wp-content/uploads/2026/01/Decreto-Supremo-001-2026-EF-LPDerecho.pdf"),
]

# (Opcional) Páginas web donde buscar MÁS PDF automáticamente (por ejemplo, una colección de gob.pe).
# El cuaderno entra a la página, encuentra los enlaces .pdf y los descarga todos.
PAGINAS_CON_PDFS = [
    # "https://www.gob.pe/institucion/oece/colecciones/45029-ley-n-32069-ley-general-de",
]
print(len(DOCUMENTOS), "documentos en la lista")

In [ ]:
# Paso 3 · Descargar (automático)
import os, re, time, requests
from urllib.parse import urljoin

CARPETA = "normativa_pdf"
os.makedirs(CARPETA, exist_ok=True)
NAVEGADOR = {"User-Agent": "Mozilla/5.0 (LiciGuard; proyecto academico)"}

def nombre_archivo(texto):
    t = re.sub(r"[^a-zA-Z0-9]+", "_", texto).strip("_").lower()
    return t[:60] + ".pdf"

def descargar(url, intentos=3):
    for i in range(intentos):
        try:
            r = requests.get(url, headers=NAVEGADOR, timeout=60)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                return r.content
            print(f"   respuesta {r.status_code}, no es PDF")
        except Exception as e:
            print(f"   intento {i+1} falló: {e}")
        time.sleep(2)
    return None

def pdfs_en_pagina(url):
    try:
        html = requests.get(url, headers=NAVEGADOR, timeout=60).text
    except Exception as e:
        print("No se pudo abrir", url, e); return []
    enlaces = re.findall(r'href=["\']([^"\']+?\.pdf)["\']', html, flags=re.I)
    return sorted({urljoin(url, e) for e in enlaces})

pendientes = list(DOCUMENTOS)
for pagina in PAGINAS_CON_PDFS:
    for enlace in pdfs_en_pagina(pagina):
        pendientes.append((os.path.basename(enlace)[:-4].replace("-", " "), enlace))

descargados = []
for nombre, url in pendientes:
    ruta = os.path.join(CARPETA, nombre_archivo(nombre))
    print("⬇️ ", nombre)
    if not os.path.exists(ruta):
        datos = descargar(url)
        if not datos:
            print("   ❌ No se pudo descargar. Revisa el enlace."); continue
        open(ruta, "wb").write(datos)
    descargados.append({"fuente": nombre, "url": url, "ruta": ruta})
    print(f"   ✅ {os.path.getsize(ruta)/1e6:.1f} MB")

# ¿Tienes PDF propios? Ponlos en la carpeta normativa_pdf (panel de archivos de Colab, a la izquierda)
for f in sorted(os.listdir(CARPETA)):
    ruta = os.path.join(CARPETA, f)
    if f.endswith(".pdf") and ruta not in {d["ruta"] for d in descargados}:
        descargados.append({"fuente": f[:-4].replace("_", " "), "url": None, "ruta": ruta})
print(f"\nTotal listos: {len(descargados)} documentos")

In [ ]:
# Paso 4 · Sacar el texto y limpiarlo (quita encabezados repetidos como "El Peruano / NORMAS LEGALES")
import pymupdf
from collections import Counter

def texto_limpio(ruta):
    with pymupdf.open(ruta) as doc:
        paginas = [p.get_text("text") for p in doc]
    lineas = Counter(l.strip() for p in paginas for l in set(p.splitlines()) if l.strip())
    repetidas = {l for l, n in lineas.items() if n >= max(3, len(paginas) * 0.3) and len(l) < 120}
    vacias = sum(len(p.strip()) < 40 for p in paginas)
    limpio = "\n".join("\n".join(l for l in p.splitlines() if l.strip() not in repetidas) for p in paginas)
    limpio = re.sub(r"-\n(?=[a-záéíóúñ])", "", limpio)   # une palabras cortadas
    return limpio, len(paginas), vacias

for d in descargados:
    d["texto"], d["paginas"], vacias = texto_limpio(d["ruta"])
    aviso = "  ⚠️ parece escaneado (sin texto)" if vacias > d["paginas"] * 0.5 else ""
    print(f"📄 {d['fuente']}: {d['paginas']} páginas, {len(d['texto']):,} caracteres{aviso}")

In [ ]:
# Paso 5 · Cortar por artículos (cada artículo será una "ficha" del RAG)
ARTICULO = re.compile(r"(?:^|\n)\s*(Art[íi]culo\s+\d+[°º]?(?:-[A-Z])?)\s*[.\-–:]+\s*", re.I)
DISPOSICION = re.compile(r"(?:^|\n)\s*((?:PRIMERA|SEGUNDA|TERCERA|CUARTA|QUINTA|SEXTA|SÉPTIMA|OCTAVA|NOVENA|DÉCIMA)[A-ZÁÉÍÓÚ ]*)\s*[.\-–]+\s*")
MAXIMO = 1800   # caracteres por ficha; los artículos largos se parten en varias

def fichas(doc):
    texto, salida = doc["texto"], []
    partes = ARTICULO.split(texto)
    if len(partes) < 3:   # sin artículos: cortar en bloques con un poco de traslape
        plano = re.sub(r"\s+", " ", texto)
        for i in range(0, len(plano), MAXIMO - 200):
            salida.append({"articulo": f"Fragmento {i // (MAXIMO - 200) + 1}", "cuerpo": plano[i:i + MAXIMO]})
    else:
        for i in range(1, len(partes) - 1, 2):
            nombre = re.sub(r"\s+", " ", partes[i]).strip().capitalize().replace("Articulo", "Artículo")
            cuerpo = re.sub(r"\s+", " ", partes[i + 1]).strip()
            for j in range(0, max(len(cuerpo), 1), MAXIMO):
                salida.append({"articulo": nombre + (f" (parte {j // MAXIMO + 1})" if j else ""),
                               "cuerpo": cuerpo[j:j + MAXIMO]})
    return [{"fuente": doc["fuente"], "articulo": s["articulo"], "titulo": s["cuerpo"].split(".")[0][:120],
             "texto": s["cuerpo"], "url": doc["url"]} for s in salida if len(s["cuerpo"]) > 30]

articulos = [f for d in descargados for f in fichas(d)]
for d in descargados:
    n = sum(a["fuente"] == d["fuente"] for a in articulos)
    print(f"{d['fuente']}: {n} fichas")
print("\nTotal:", len(articulos), "fichas")
print("\nEjemplo:", articulos[0]["articulo"] if articulos else "-", "→", articulos[0]["texto"][:200] if articulos else "")

In [ ]:
# Paso 6 · Probar el buscador (así buscará la app). Cambia las preguntas si quieres.
import math
def palabras(t):
    t = t.lower().translate(str.maketrans("áéíóúü", "aeiouu"))
    return [w[:7] for w in re.findall(r"[a-zñ0-9]+", t) if len(w) > 2]
docs = [Counter(palabras(a["articulo"] + " " + a["texto"])) for a in articulos]
df = Counter(w for d in docs for w in d)
def buscar(q, k=3):
    puntajes = []
    for i, d in enumerate(docs):
        s = sum(math.log(1 + len(docs) / df[w]) * d[w] / (d[w] + 1.5) for w in palabras(q) if w in d)
        puntajes.append((s, i))
    return [articulos[i] for s, i in sorted(puntajes, reverse=True)[:k] if s > 0]

for pregunta in ["¿se puede exigir una marca?", "experiencia del postor", "penalidad por mora", "garantía de fiel cumplimiento"]:
    print("\n🔎", pregunta)
    for a in buscar(pregunta, 2):
        print(f"   {a['fuente']} · {a['articulo']}: {a['texto'][:120]}…")

In [ ]:
# Paso 7 · Guardar y descargar articulos.json
import json
with open("articulos.json", "w", encoding="utf-8") as f:
    json.dump(articulos, f, ensure_ascii=False, indent=1)
print(f"articulos.json creado: {len(articulos)} fichas, {os.path.getsize('articulos.json')/1e6:.1f} MB")
try:
    from google.colab import files
    files.download("articulos.json")
    print("Descargando… guárdalo en la carpeta data/normativa/ de tu proyecto.")
except ImportError:
    print("Copia articulos.json a la carpeta data/normativa/ de tu proyecto.")

## Listo 🎉

1. Pon `articulos.json` en `data/normativa/` de tu proyecto.
2. Reinicia la app (`Ctrl + C` y `streamlit run app.py`).
3. En la barra lateral verás **"Normativa: N artículos"**. Desde ahora:
   - cada señal de **¿Es justa?** muestra el artículo relacionado,
   - el **asistente** responde usando la ley y tus bases.

**¿Quieres agregar más conocimiento?** Añade enlaces en el Paso 2 (opiniones del OECE, la directiva de bases estándar, pronunciamientos) y vuelve a ejecutar todo.